# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushdevo/10x.ai/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
## Finding 1
The FlyRank paper reports that machine learning models can identify customers at risk of churn before they leave.

### Methodology Question
Where does the churn label come from? If churn is defined using future customer behavior, it is important to ensure that no future information leaks into training features.

### Validation Concern
The claim is reasonable if the validation split prevents the same customer from appearing in both training and validation sets. A random row split may overestimate performance because the model can learn customer-specific patterns.

### Constructive Assessment
I would prefer a grouped validation design using customer IDs so that the reported performance better reflects real-world deployment.
## Finding 2
The paper reports strong predictive performance using behavioral features such as transaction counts and engagement metrics.

### Methodology Question
Are all behavioral features available at prediction time? Some aggregated features may accidentally include future events.

### Validation Concern
High accuracy alone does not prove generalization. If features are calculated using information from periods after the prediction date, the model may benefit from leakage.

### Constructive Assessment
A time-aware validation split would strengthen the claim because it tests whether the model can predict future churn using only historical information.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

# Original random split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = CatBoostClassifier(verbose=0)
model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:,1]
random_auc = roc_auc_score(y_test, preds)

print("Random Split AUC:", random_auc)
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = CatBoostClassifier(verbose=0)
model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:,1]
group_auc = roc_auc_score(y_test, preds)

print("Grouped Split AUC:", group_auc)

| Validation Strategy | AUC |
|--------------------|------|
| Random Split | 0.89 |
| Grouped Split | 0.82 |

The grouped split produced lower performance. This suggests that some of the original performance may have benefited from customer overlap between train and test sets. The grouped result is likely a more realistic estimate of deployment performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
leakage_candidates = [
    "future_revenue",
    "future_sessions",
    "days_until_churn",
    "target_encoded_feature",
    "label_proxy"
]

for col in leakage_candidates:
    if col in df.columns:
        print("Potential leakage feature:", col)

| Feature         | Risk      | Reason                                |
|-----------------|-----------|---------------------------------------|
| impressions_90d | Low       | Historical activity                   |
| clicks_90d      | Low       | Historical activity                   |
| sessions_90d    | Low       | Available at prediction time          |
| days_until_churn| High      | Uses future information               |
| future_revenue  | High      | Not available when prediction is made |



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
The observed AUC suggests directional predictive value for churn risk ranking. Further validation on future data would be needed before operational deployment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.